In [1]:
import requests
import pandas as pd
import time
from io import BytesIO
from datetime import datetime

import sys
sys.path.append('..')
from utils.bucket_utils import get_duck_con

In [2]:
import os
!{sys.executable} -m pip install playwright pandas pyarrow fastparquet
!{sys.executable} -m playwright install chromium

  Using cached playwright-1.58.0-py3-none-macosx_11_0_arm64.whl (41.0 MB)
  Using cached pyarrow-21.0.0-cp39-cp39-macosx_12_0_arm64.whl (31.2 MB)
  Using cached fastparquet-2024.11.0-cp39-cp39-macosx_11_0_arm64.whl (684 kB)
  Using cached greenlet-3.2.5-cp39-cp39-macosx_11_0_universal2.whl (274 kB)
  Using cached pyee-13.0.1-py3-none-any.whl (15 kB)
  Using cached cramjam-2.11.0-cp39-cp39-macosx_11_0_arm64.whl (1.7 MB)
You should consider upgrading via the '/Users/palmchns/113-pea-oms/.venv/bin/python -m pip install --upgrade pip' command.


In [ ]:
async def main():
    print("🚀 เริ่มกระบวนการดึงข้อมูล...")
    df_new = await scrape_pathum_all_districts()

    # 1. เตรียมข้อมูลวันที่และเวลาสำหรับชื่อไฟล์
    now = datetime.now()
    today_real_date = get_today_thai_format() # "26 มีนาคม 2569"
    timestamp = now.strftime("%Y%m%d_%H%M")   # "20260326_1740"

    # 2. กำหนด Path โฟลเดอร์
    desktop_path = os.path.expanduser("~/Desktop")
    target_folder = os.path.join(desktop_path, "JS100")

    if not os.path.exists(target_folder):
        os.makedirs(target_folder)
        print(f"📁 สร้างโฟลเดอร์ใหม่ที่: {target_folder}")

    # 3. จัดการข้อมูล
    if not df_new.empty:
        # กรองเอาเฉพาะของวันนี้
        df_today = df_new[df_new['วันที่'] == today_real_date].copy()
        
        if df_today.empty:
            print(f"⚠️ ไม่พบรายการของวันที่ {today_real_date} (ข้ามการบันทึก)")
        else:
            file_name = os.path.join(target_folder, f"JS100_Pathum_{timestamp}.parquet")
            
            # บันทึกไฟล์
            df_today.to_parquet(file_name, engine='fastparquet', index=False)
            
            print(f"✅ บันทึกข้อมูลสำเร็จ!")
            print(f"📄 ชื่อไฟล์: JS100_Pathum_{timestamp}.parquet")
            print(f"📍 ที่อยู่: {file_name}")
            
            # 💡 เปลี่ยนจาก display เป็น print เพื่อไม่ให้ Error นอก Jupyter
            print(df_today)
    else:
        print("🤷‍♂️ ไม่พบข้อมูลใหม่จากการค้นหาครั้งนี้")

    print("✅ กระบวนการเสร็จสิ้น!")

# --- จุดสตาร์ทของระบบ ---
if __name__ == "__main__":
    table_today = await main()

In [ ]:
display(table_today)